In [3]:
!pip install fusion_solar_py -q

In [4]:
from fusion_solar_py.client import FusionSolarClient,logged_in
import json
from kaggle_secrets import UserSecretsClient


In [5]:
class FusionSolarClientExtended(FusionSolarClient):
    """Extension of FusionSolarClient with additional functionality."""

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    MAXIMUM_SELF_CONSUMPTION = 2
    FULLY_FEED_TO_GRID = 4

    @logged_in
    def set_battery_working_mode(self, battery_id, mode_id):
        valid_modes = [self.MAXIMUM_SELF_CONSUMPTION, self.FULLY_FEED_TO_GRID]
        if mode_id not in valid_modes:
            raise ValueError(f"Invalid mode_id: {mode_id}. Expected one of {valid_modes}.")
    
        url = f"https://{self._huawei_subdomain}.fusionsolar.huawei.com/rest/pvms/web/device/v1/deviceExt/set-config-signals"
        data = {
            "dn": battery_id,
            "changeValues": f'[{{"id":"230320241","value":"{mode_id}"}}]',
        }

        response = self._session.post(url, data=data)
        response.raise_for_status()

        try:
            response_json = response.json()
            print(f"Response: {response_json}")
        except ValueError:
            print("Error: The response is not in JSON format.")

In [ ]:
user_secrets = UserSecretsClient()
FUSION_SOLAR_CLIENT_PASSWORD = user_secrets.get_secret("FUSION_SOLAR_CLIENT_PASSWORD")
FUSION_SOLAR_CLIENT_USERNAME = user_secrets.get_secret("FUSION_SOLAR_CLIENT_USERNAME")

In [9]:
# log into the API - with proper credentials...
client = FusionSolarClientExtended(FUSION_SOLAR_CLIENT_USERNAME, FUSION_SOLAR_CLIENT_PASSWORD,huawei_subdomain="uni004eu5")
plant_id=client.get_plant_ids()[0]
battery_id=client.get_battery_ids(plant_id)[0]

In [10]:
battery_id

'NE=150331872'

In [8]:
client.set_battery_working_mode(battery_id,2)

NameError: name 'battery_id' is not defined